In [1]:
import os
import torch
import torchvision

device = torch.device("cuda")

def torch_ds():
    input_resolution = 224
    data = "/data/ImageNet/"
    value_range = v2.Normalize(
        mean=[0.5] * 3,
        std=[0.5] * 3
    )
    
    val_dataset = datasets.ImageNet(
        data,
        split='val',
        transform=v2.Compose([
            v2.ToImage(),
            v2.Resize(256),
            v2.CenterCrop(input_resolution),
            v2.ToDtype(torch.float32, scale=True),
            value_range,
        ])
    )

    batch_size = 64

    val_loader = torch.utils.data.DataLoader(
        val_dataset, batch_size=batch_size, shuffle=False,
        num_workers=4, pin_memory=True, sampler=None,
        multiprocessing_context='spawn'
    )
    return val_loader

from simple_vit import SimpleVisionTransformer

hidden_dim = 384
input_resolution = 224
vit = SimpleVisionTransformer(
	image_size=input_resolution,
	patch_size=16,
	num_layers=12,
	num_heads=6,
	hidden_dim=hidden_dim,
	mlp_dim=hidden_dim * 4,
	representation_size=hidden_dim,
)

ROOT = "/media/jason-chou/T31/imagenet-runs/logs"
run = "better-baseline-tf-300ep-0.05-lower-scale-0"
p = os.path.join(ROOT, run)

best_path = os.path.join(p, 'checkpoints/model_best.pth.tar')
ckpt = torch.load(best_path, weights_only=True)
state = ckpt['state_dict']

try:
    vit.load_state_dict(state)
except:
    vit.load_state_dict({k[len('module.'):]: v for k, v in state.items()})

In [7]:
def grad(vit, images, target):
    images.requires_grad = True
    logits, _ = vit(images, 1.0, target, target)
    target_logits = logits[torch.arange(logits.shape[0]), target]
    total = sum(target_logits)
    total.backward()
    grad_sum_clipped = torch.clamp(images.grad.sum(dim=1), min=0.)
    return grad_sum_clipped

In [3]:
vit = vit.to(device)

In [4]:
import torchvision.datasets as datasets
from torchvision.transforms import v2

import matplotlib.pyplot as plt
import numpy as np
import cv2

# create heatmap from mask on image
def show_cam_on_image(img, mask):
    heatmap = cv2.applyColorMap(np.uint8(255 * mask), cv2.COLORMAP_JET)
    heatmap = np.float32(heatmap) / 255
    cam = heatmap + np.float32(img)
    cam = cam / np.max(cam)
    return cam

data = "/data/ImageNet/"

value_range = v2.Normalize(
    mean=[0.5] * 3,
    std=[0.5] * 3
)

transform = v2.Compose([
    v2.ToImage(),
    v2.Resize(256),
    v2.CenterCrop(input_resolution),
    v2.ToDtype(torch.float32, scale=True),
    value_range,
])

val_dataset = datasets.ImageNet(
    data,
    split='val',
    transform=transform
)

first_25_classes = torch.utils.data.Subset(val_dataset, torch.arange(0, 1250, 50))
loader = torch.utils.data.DataLoader(
    first_25_classes, batch_size=25, shuffle=False,
    num_workers=1, pin_memory=True, sampler=None,
    multiprocessing_context='spawn'
)

In [9]:
images, target = next(iter(loader))

images = images.cuda()
target = target.cuda()
cam = grad(vit, images, target)

low, high = cam.amin(dim=(1, 2), keepdim=True), cam.amax(dim=(1, 2), keepdim=True)
cam_normalized = (cam - low) / (high - low)
cam_normalized = cam_normalized.detach()
original_images = (images / 2 + 0.5).detach()

fig, axs = plt.subplots(5, 5, figsize=(100, 100))

for i in range(5):
    for j in range(5):
        index = i * 5 + j
        image_transformer_attribution = original_images[index].permute(1, 2, 0).data.cpu().numpy()
        image_transformer_attribution = (image_transformer_attribution - image_transformer_attribution.min()) / (image_transformer_attribution.max() - image_transformer_attribution.min())
        
        vis = cam_normalized[index].reshape(224, 224).data.cpu().numpy()
        vis = show_cam_on_image(image_transformer_attribution, vis)
        vis =  np.uint8(255 * vis)
        vis = cv2.cvtColor(np.array(vis), cv2.COLOR_RGB2BGR)
        axs[i, j].imshow(vis)
        axs[i, j].axis('off')

plt.savefig("grad_first_25_classes.png")